(user_guide_optimization_aco)=

# Ant Colony Optimization (ACO)

📖 {ref}`Optimization API Reference <api_optimization_ACO>`

**Ant Colony Optimization (ACO)** is a metaheuristic inspired by how real ant colonies find
short paths between their nest and a food source. A single ant explores more or less at
random, but it deposits a chemical trail (pheromone) as it moves. Ants that happen to take a
shorter path complete more round trips per unit time, so their pheromone accumulates faster.
Because ants are more likely to follow trails with stronger pheromone, short paths get
reinforced, and over many iterations the colony's collective behavior converges toward good
paths -- without any single ant "knowing" the best route.

Algorithmically, every ACO variant repeats the same three-step loop:

1. **Construct solutions** -- each of `nAnts` ants builds a candidate solution, using the
   current pheromone information to bias (but not fully determine) its choices.
2. **Evaluate** the constructed solutions with the objective function.
3. **Update the pheromones** -- existing pheromone evaporates a little (so the colony can
   forget bad decisions and keep exploring), and the ants that constructed good solutions
   deposit new pheromone that reinforces the choices they made.

This loop is implemented once, in {py:class}`~sigmaepsilon.math.optimize.aco.AntColonyOptimization`,
which also takes care of bookkeeping shared by all variants: tracking the best solution found so
far (the *champion*), counting iterations and function evaluations, and stopping either after
`maxiter` iterations or once the champion hasn't improved for `maxage` iterations in a row
(stagnation). What differs between problem types is *what* a "solution" is and *how* pheromones
are represented, so the base class leaves `construct_solutions` and `update_pheromones` abstract,
and two concrete subclasses fill them in:

* {py:class}`~sigmaepsilon.math.optimize.aco.ContinuousAntColonyOptimization` (**ACOR**), for
  unconstrained-shape, box-constrained problems over continuous real-valued variables.
* {py:class}`~sigmaepsilon.math.optimize.aco.CombinatorialAntColonyOptimization` (**Ant System**),
  for permutation problems defined over a distance matrix, the classic example being the
  Traveling Salesman Problem (TSP).

## When to use it (and when not to)

ACO is a good fit when:

* the search space is not smooth enough (or not known in closed form) for gradient-based
  methods, so you need a derivative-free, population-based search, and
* the problem has either a natural continuous, box-constrained structure (use ACOR), or a
  natural *sequencing / permutation* structure with a notion of pairwise cost between elements
  (use the combinatorial variant) -- routing, scheduling, assignment-like problems.

It is usually **not** the right tool when:

* the objective is linear (or convex) and constraints are linear -- use
  [Linear Programming](lp.ipynb) (`linprog`/HIGHS), which will find the *exact* optimum, faster and
  deterministically.
* the problem is smooth and gradient information is available -- a gradient-based nonlinear
  solver will typically converge faster and more reliably than any metaheuristic.
* you need a real-valued box-constrained search but don't have a combinatorial structure -- the
  library's [genetic algorithms](bga.ipynb) solve the same class of problems as
  ACOR and are worth comparing against; neither is strictly better; ACOR tends to do well when
  good solutions cluster together in a few regions, while GAs with crossover can be better at
  combining unrelated partial solutions.

As with any metaheuristic, ACO gives you *no optimality guarantee* -- only a good solution found
within a finite budget of iterations and function evaluations.

## Continuous ACO (ACOR)

`ContinuousAntColonyOptimization` implements ACOR (Socha & Dorigo, 2008). Instead of a
pheromone matrix, it keeps a **solution archive**: the `archive_size` best solutions found so
far, sorted by fitness. In each iteration:

* every ant picks one archive member as a "seed", with better-ranked members picked with higher
  probability (the `q` parameter controls how strongly the choice is biased toward the best
  solutions -- small `q` means an almost greedy pick, large `q` gives near-uniform choice among
  archive members),
* the ant then samples a new candidate solution from a Gaussian centered on that seed, with a
  per-dimension standard deviation derived from how spread out the archive is along that
  dimension, scaled by `xi` (larger `xi` means wider sampling, i.e. more exploration),
* the newly-constructed solutions are merged into the archive together with the previous
  members, and only the `archive_size` best survive -- this *is* the pheromone update: the
  archive itself plays the role of the pheromone trail, since it is what future iterations
  sample around.

So the archive both memorizes the current best-known solutions and, through its spread,
controls how far new solutions can stray from them -- shrinking automatically as the population
converges.

In [ ]:
from sigmaepsilon.math.optimize import ContinuousAntColonyOptimization as ACOR


def rosenbrock(x, a=1.0, b=100.0):
    return (a - x[0]) ** 2 + b * (x[1] - x[0] ** 2) ** 2


ranges = [[-5, 5], [-5, 5]]  # box constraints for x0 and x1

acor = ACOR(
    rosenbrock,
    ranges,
    archive_size=30,
    nAnts=20,
    q=0.3,
    xi=0.7,
    maxiter=200,
    minimize=True,
    seed=42,
)
result = acor.solve()
result.phenotype, result.fitness

The Rosenbrock function has its minimum at $(1, 1)$ with value $0$; the result above should
land close to it. `solve` returns the champion as an
{py:class}`~sigmaepsilon.math.optimize.aco.AntSolution` (with `.phenotype` and `.fitness`); the
same information is also available afterwards via `acor.best_phenotype()` /
`acor.best_candidate()`, and `acor.state` exposes the run's iteration/evaluation counters.

In [ ]:
acor.state.n_iter, acor.state.n_fev, acor.state.success

If your objective function can evaluate a whole batch of candidates at once (e.g. vectorized
NumPy code), pass `vectorized=True` so all `nAnts` candidates of an iteration are evaluated in a
single call instead of one Python-level call per ant; for expensive, non-vectorizable objectives
you can instead parallelize across ants with `n_jobs` (`-1` uses all available CPUs).

## Combinatorial ACO (Ant System / TSP)

`CombinatorialAntColonyOptimization` implements the original Ant System algorithm (Dorigo et
al., 1996) for problems defined over a symmetric `distance_matrix`, most naturally the
Traveling Salesman Problem: visit every node exactly once and return to the start, minimizing
total travel distance.

Here the pheromone really is a matrix $\tau_{ij}$, one entry per directed edge between nodes,
initialized to `tau_init`. Alongside it, a fixed heuristic desirability $\eta_{ij} = 1 /
d_{ij}$ favors short edges regardless of pheromone. Each ant builds a full tour node by node,
starting from a random node and, at every step, picking the next unvisited node $j$ from the
current node $i$ with probability proportional to

$$
\tau_{ij}^{\alpha} \cdot \eta_{ij}^{\beta}
$$

so `alpha` controls how strongly the ants trust accumulated pheromone (experience) and `beta`
controls how strongly they trust the raw distance (greediness). After all ants have built a
tour, pheromone evaporates everywhere by a factor `(1 - rho)`, and every ant deposits
`Q / tour_length` on each edge of its tour -- so shorter tours reinforce their edges more
strongly, and edges nobody uses keep fading away.

In [ ]:
import numpy as np
from sigmaepsilon.math.optimize import CombinatorialAntColonyOptimization as AntSystem

rng = np.random.default_rng(0)
n_cities = 15
coords = rng.uniform(0, 100, size=(n_cities, 2))
distance_matrix = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)


def tour_length(tour):
    return sum(
        distance_matrix[tour[i], tour[(i + 1) % len(tour)]] for i in range(len(tour))
    )


aco = AntSystem(
    tour_length,
    distance_matrix,
    alpha=1.0,
    beta=3.0,
    rho=0.3,
    nAnts=30,
    maxiter=150,
    minimize=True,
    seed=0,
)
result = aco.solve()
result.phenotype, result.fitness

`result.phenotype` is the visiting order of the cities; let's plot the resulting tour.

In [ ]:
import matplotlib.pyplot as plt

tour = [int(i) for i in result.phenotype] + [int(result.phenotype[0])]
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(coords[tour, 0], coords[tour, 1], "o-", color="mediumblue")
for i, (x, y) in enumerate(coords):
    ax.annotate(str(i), (x, y), textcoords="offset points", xytext=(5, 5))
ax.set_title(f"Best tour found (length = {result.fitness:.1f})")
ax.set_aspect("equal")
plt.show()

## A note on parameters

Both variants share `nAnts`, `maxiter`, `miniter`, `rho`, `maxage` and `minimize` /
`seed` from the base class; `rho` is unused by ACOR (kept only for a uniform constructor
signature) since ACOR's "pheromone update" is the archive replacement described above, not an
evaporation/deposit rule. There is no universally correct parameter set -- as with any
metaheuristic, expect to tune `nAnts`/`maxiter` against your evaluation budget, and the
exploration/exploitation knobs (`q`, `xi` for ACOR; `alpha`, `beta`, `rho` for Ant System)
against how rugged your search space is. Setting `seed` makes runs reproducible, which is
useful while tuning.